# YOLO, LocateAnything Inference & Latency Logging · Edge-Computing Notebook

Edge AI means running models **on the device**, close to where data is produced, instead of sending everything to the cloud. In this lab you will run object-detection inference on the DGX Spark with two kinds of model, and — just as importantly — **measure how long inference takes** and log it in a structured form.

```text
Image  ->  YOLO (fixed classes)            ->  detections + latency
Image  ->  LocateAnything (text prompt)    ->  detections + latency
            -> JSON-lines latency log on the edge node
```

Two model styles:

```text
YOLO            -> fast, detects a fixed set of trained classes (person, car, ...)
LocateAnything  -> open-vocabulary: you give it a text prompt and it locates what you describe
```

The latency log you produce here uses the same JSON-lines format as your telemetry, and it is the direct input for the next lab (Benchmarking & Evaluations).

**You will:**
- run YOLO inference on CPU and GPU and compare the reported speeds
- log per-inference latency (preprocess / inference / postprocess / end-to-end) as JSON lines
- run an open-vocabulary "locate anything" model steered by a text prompt
- capture a sustained stream log while watching device load with `tegrastats`

The main idea: at the edge, a model is only useful if it is both **accurate enough** and **fast enough**. This lab is where you start measuring "fast enough."

## How this notebook works · cells and a Jupyter terminal

Most steps here run once and return (one inference, a 50-run latency log) — those are ordinary notebook cells. A few steps run live and never return on their own, like watching `tegrastats` stream device load. For those, this notebook **writes a small script into your lab folder and asks you to run it in a Jupyter terminal.**

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Terminal]** · open a Jupyter terminal — **File ▸ New ▸ Terminal** (or the Terminal tile in the Launcher) — and run the given command. The terminal is a real shell **on the DGX Spark**.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So the setup cell below writes `~/edgeAILab/labEnv.sh` with your `DEVICE_ID` and model names, and every helper script starts with `source ~/edgeAILab/labEnv.sh` — the notebook and your terminals always agree.

**Requirements:** the notebook kernel must have `ultralytics` importable (instructor container, or the venv described in Part 1). GPU runs need a Jetson-matched PyTorch build.

In [1]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

labHelpers ready - setupLab, preflight, checkpoint, labSummary, showFile, showScriptCard, showEnvCard, showNote, dockerVersions, dockerPs, dockerLogs + check predicates


---
## Part 0 · Before You Begin

You are working on a shared **DGX Spark** edge device, through JupyterLab running on the DGX Spark itself. **Every command in this notebook runs on the DGX Spark**, not on your own computer.

Because the DGX Spark is shared:

- use your own home folder and keep your files under it
- prefix any shared resource names with your account name using `${USER}`
- the GPU is shared too — if inference is slow, another student may be running a job; coordinate with your instructor

> If you prefer SSH: `ssh <your-account>@<thor-address>` gives you the same shell a Jupyter terminal does.

---
## Part 1 · Set Up Your Working Folder and Inference Environment

**[Notebook cell]** Create the lab identity. This lab exposes no network service, so no unique port matters here — `setupLab` still derives a default `PORT`, which simply goes unused. `DEVICE_ID` tags every latency record with *your* device identity:

In [2]:
import os

labConfig = setupLab(
    "edgeAILab",
    extraEnv={
        "DEVICE_ID": f"{os.environ.get('USER', 'student01')}-{deviceName()}",
        "MODEL": "yolov8n.pt",
        # Open-vocabulary model for Part 6. Replace with the model or weights
        # your instructor provides, if different.
        "LOCATE_MODEL": "yolov8s-world.pt",
    },
)

This lab uses the Ultralytics YOLO package for inference. There are two common ways to get a working environment on the DGX Spark:

```text
Option A: a prebuilt container the instructor provides (CUDA + Ultralytics already installed)
Option B: a Python virtual environment you create yourself
```

**Your instructor will tell you which option to use.** Edge AI dependencies (PyTorch, CUDA) must match the Jetson software stack, so a provided container is often the most reliable path.

For **Option B**, create the venv and register it as a Jupyter kernel in a **[Terminal]**, then switch this notebook to it (**Kernel ▸ Change Kernel ▸ Python (edge-ai venv)**) and re-run from the top:

```bash
python3 -m venv ~/edgeAILab/venv
source ~/edgeAILab/venv/bin/activate
pip install ultralytics ipykernel
python -m ipykernel install --user --name edgeAIVenv --display-name "Python (edge-ai venv)"
```

> On a Jetson, the GPU build of PyTorch must match the installed JetPack/L4T version. If `pip install ultralytics` pulls a CPU-only or mismatched PyTorch, GPU inference will not work. Use the wheel or container your instructor provides for GPU support.

### Preflight · check your environment

In [3]:
import os

def cudaAvailableProbe():
    try:
        import torch
        cudaOk = torch.cuda.is_available()
        return cudaOk, f"torch.cuda.is_available() = {cudaOk}"
    except Exception as torchError:
        return False, str(torchError)[:100]

preflight(
    [
        check("ultralytics importable", pythonImportable("ultralytics"),
              hint="Switch this notebook's kernel to the instructor container / edge-ai venv (Part 1, Option A/B), "
                   "or install with: pip install ultralytics.",
              link="https://docs.ultralytics.com/quickstart/"),
        check("torch importable", pythonImportable("torch"),
              hint="PyTorch on a Jetson must match the JetPack/L4T version - use the wheel or container "
                   "your instructor provides.",
              link="https://developer.nvidia.com/embedded/jetpack"),
        check("CUDA visible to torch (GPU runs)", cudaAvailableProbe,
              hint="GPU inference needs a CUDA-enabled, Jetson-matched PyTorch. The CPU parts of this lab "
                   "still work without it.",
              link="https://docs.ultralytics.com/guides/nvidia-jetson/"),
        check("GPU telemetry on PATH (Part 7)", commandOnPath("nvidia-smi"),
              hint="tegrastats ships with JetPack (usually /usr/bin/tegrastats). Only the Part 7 load watcher "
                   "needs it; everything else works without.",
              link="https://docs.nvidia.com/jetson/archives/r36.3/DeveloperGuide/SD/PlatformPowerAndPerformance/JetsonOrinNanoSeriesJetsonOrinNxSeriesAndJetsonAgxOrinSeries.html"),
    ],
    infoRows=[
        ("your USER", os.environ.get("USER", "?")),
        ("DEVICE_ID", os.environ.get("DEVICE_ID", "?")),
        ("default MODEL", os.environ.get("MODEL", "?")),
        ("LOCATE_MODEL", os.environ.get("LOCATE_MODEL", "?")),
    ],
)

                              preflight - environment                              
                                                                                   
  check                              result     detail                             
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  ultralytics importable             ✗ failed   No module named 'ultralytics'      
  torch importable                   ✓ ok       import torch ok                    
  CUDA visible to torch (GPU runs)   ✗ failed   torch.cuda.is_available() = False  
  tegrastats on PATH (Part 7)        ✗ failed   tegrastats not on PATH             
  your USER                          info       papka                              
  DEVICE_ID                          info       papka-thor                         
  default MODEL                      info       yolov8n.pt                         
  LOCATE_MODEL                       info       yolov8s-world.pt

False

**[Notebook cell]** Create the data folder, move into the lab folder, and verify the package imports:

In [4]:
!mkdir -p ~/edgeAILab/data
%cd ~/edgeAILab

/Users/papka/edgeAILab


In [5]:
import ultralytics
print("ultralytics", ultralytics.__version__)

ModuleNotFoundError: No module named 'ultralytics'

**You should see:** `ultralytics` followed by a version number, with no traceback.

> **If it doesn't work:** `ModuleNotFoundError: ultralytics` means this kernel is not the inference environment — switch to the instructor's container kernel (Option A) or the edge-ai venv kernel (Option B) and re-run from the top. A `torch`/CUDA error here is the version-mismatch warning above: use the instructor-provided build.

### Checkpoint · Part 1

In [ ]:
checkpoint(
    "Part 1 - working folder and environment",
    [
        check("~/edgeAILab/data exists", dirExists("~/edgeAILab/data"),
              hint="Re-run the mkdir cell above.",
              link="https://docs.python.org/3/library/pathlib.html"),
        check("DEVICE_ID exported", envVarSet("DEVICE_ID"),
              hint="Re-run the setupLab cell at the top of Part 1.",
              link="https://docs.python.org/3/library/os.html#os.environ"),
        check("ultralytics importable", pythonImportable("ultralytics"),
              hint="Fix the environment first (Part 1, Option A/B) - every later part depends on it.",
              link="https://docs.ultralytics.com/quickstart/"),
    ],
    successNote="Folder, identity, and inference environment are ready.",
    docLink="https://docs.ultralytics.com/",
)

---
## Part 2 · What Inference and Latency Mean

**Inference** is running a trained model on new input to get a prediction. For object detection, the input is an image and the output is a set of boxes with labels and confidence scores.

**Latency** is how long one inference takes. It is not a single number; it breaks into stages:

```text
preprocess   -> resize / normalize the image into the model's input format
inference    -> the model forward pass (this is what the GPU accelerates)
postprocess  -> turn raw model output into boxes (filtering, non-max suppression)
end_to_end   -> the total time you actually wait, from input to result
```

Two related but different goals:

```text
Latency    -> how long a single result takes (matters for real-time response)
Throughput -> how many results per second (frames per second, FPS)
```

At the edge these trade off against each other, and against accuracy and power. You cannot improve what you do not measure — so you will measure all of it.

---
## Part 3 · Run Your First YOLO Inference

Ultralytics ships small sample images, so you can run a detection with no extra downloads.

**[Notebook cell]** Create `detectOnce.py` (the original lab uses `nano`; `%%writefile` does the same job):

In [ ]:
%%writefile detectOnce.py
import os
from ultralytics import YOLO
from ultralytics.utils import ASSETS

modelName = os.environ.get("MODEL", "yolov8n.pt")
deviceName = os.environ.get("DEVICE", "cpu")  # "cpu" or "0" for the first GPU
imagePath = os.environ.get("IMAGE", str(ASSETS / "bus.jpg"))

model = YOLO(modelName)
results = model(imagePath, device=deviceName, verbose=True)

result = results[0]
print(f"\nimage: {imagePath}")
print(f"detections: {len(result.boxes)}")
print(f"reported speed (ms): {result.speed}")

# save an annotated copy so you can see the boxes
outPath = "data/detectOnce.jpg"
result.save(filename=outPath)
print(f"annotated image saved to {outPath}")

In [ ]:
# Preview the script we just wrote.
showFile("~/edgeAILab/detectOnce.py")

**[Notebook cell]** Run it on the CPU first. The first run downloads the `yolov8n.pt` weights if they are not already present (your instructor may pre-stage these if the DGX Spark has limited internet):

In [ ]:
!DEVICE=cpu python3 detectOnce.py

You should see a detection count and a `speed` dictionary with `preprocess`, `inference`, and `postprocess` times in milliseconds. The original lab copies the annotated image back to your laptop with `scp` — in a notebook we can simply display it inline:

In [ ]:
import pathlib
from IPython.display import Image as ipyImage, display

annotatedPath = pathlib.Path.home() / "edgeAILab" / "data" / "detectOnce.jpg"
if annotatedPath.exists():
    display(ipyImage(filename=str(annotatedPath), width=480))
else:
    showNote("data/detectOnce.jpg not found - run the detectOnce.py cell above first.", kind="warn")

**[Notebook cell]** Now run the same thing on the GPU (use `0` for the first GPU):

In [ ]:
!DEVICE=0 python3 detectOnce.py

Compare the `inference` time between the two runs. The GPU should be substantially faster for the forward pass.

> **New term · inference:** running a trained model on new input to get a prediction. The `speed` dict breaks one inference into `preprocess` (prep the image), `inference` (the model's forward pass — what the GPU accelerates), and `postprocess` (turn raw output into boxes), all in milliseconds.

### Checkpoint · Part 3

In [ ]:
checkpoint(
    "Part 3 - first YOLO inference",
    [
        check("detectOnce.py exists", fileExists("~/edgeAILab/detectOnce.py"),
              hint="Re-run the %%writefile detectOnce.py cell above.",
              link="https://docs.ultralytics.com/modes/predict/"),
        check("annotated image written", fileNonEmpty("~/edgeAILab/data/detectOnce.jpg"),
              hint="Re-run `!DEVICE=cpu python3 detectOnce.py`. If the weights download failed, ask the "
                   "instructor for the pre-staged yolov8n.pt.",
              link="https://docs.ultralytics.com/modes/predict/#working-with-results"),
    ],
    successNote="You ran object detection on the edge device and saw the per-stage speed report.",
    docLink="https://docs.ultralytics.com/modes/predict/",
)

---
## Part 4 · Log Inference Latency to JSON Lines

A single run is not enough to judge speed: the first inference includes one-time setup, and times vary run to run. Build a logger that **warms up**, runs many inferences, and records each one as a structured JSON line.

**[Notebook cell]** Create `latencyLogger.py`:

In [ ]:
%%writefile latencyLogger.py
import os
import json
import time
from datetime import datetime, timezone
from ultralytics import YOLO
from ultralytics.utils import ASSETS

modelName = os.environ.get("MODEL", "yolov8n.pt")
deviceName = os.environ.get("DEVICE", "cpu")        # "cpu" or "0"
deviceID = os.environ.get("DEVICE_ID", deviceName())
imagePath = os.environ.get("IMAGE", str(ASSETS / "bus.jpg"))
runCount = int(os.environ.get("RUNS", "50"))
warmupCount = int(os.environ.get("WARMUP", "5"))
logFile = os.environ.get("LOG_FILE", "data/latency.jsonl")

computeKind = "gpu" if deviceName != "cpu" else "cpu"
model = YOLO(modelName)

# warmup runs are NOT logged: the first inferences include loading and one-time setup
for _ in range(warmupCount):
    model(imagePath, device=deviceName, verbose=False)

os.makedirs(os.path.dirname(logFile) or ".", exist_ok=True)
print(f"logging {runCount} runs of {modelName} on {computeKind} -> {logFile}", flush=True)

with open(logFile, "a") as outFile:
    for runIndex in range(runCount):
        startTime = time.perf_counter()
        results = model(imagePath, device=deviceName, verbose=False)
        endTime = time.perf_counter()

        result = results[0]
        speed = result.speed  # milliseconds, reported by Ultralytics

        record = {
            "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "device_id": deviceID,
            "model": os.path.basename(str(modelName)),
            "task": "detect",
            "compute": computeKind,
            "run": runIndex,
            "latency_ms": {
                "preprocess": round(speed.get("preprocess", 0.0), 2),
                "inference": round(speed.get("inference", 0.0), 2),
                "postprocess": round(speed.get("postprocess", 0.0), 2),
                "end_to_end": round((endTime - startTime) * 1000, 2),
            },
            "num_detections": len(result.boxes),
        }

        line = json.dumps(record)
        print(line, flush=True)
        outFile.write(line + "\n")
        outFile.flush()

print("done", flush=True)

In [ ]:
# Preview the logger we just wrote.
showFile("~/edgeAILab/latencyLogger.py", maxLines=30)

The record keys (`device_id`, `latency_ms.end_to_end`, ...) are the data schema the Benchmarking lab expects — keep them exactly as written.

**[Notebook cell]** Run it on the GPU (50 runs, about a minute):

In [ ]:
!DEVICE=0 RUNS=50 LOG_FILE=data/latencyGPU.jsonl python3 latencyLogger.py

**[Notebook cell]** Then on the CPU (this takes noticeably longer — that is the point):

In [ ]:
!DEVICE=cpu RUNS=50 LOG_FILE=data/latencyCPU.jsonl python3 latencyLogger.py

**[Notebook cell]** Check the logs:

In [ ]:
!wc -l data/latencyGPU.jsonl data/latencyCPU.jsonl
!tail -n 3 data/latencyGPU.jsonl

You now have per-inference latency records for two compute configurations. The next lab turns these into statistics, but you can already eyeball the difference between CPU and GPU `inference` times.

### Checkpoint · Part 4

In [ ]:
requiredRecordKeys = ["timestamp", "device_id", "model", "task", "compute", "run",
                      "latency_ms", "num_detections"]

checkpoint(
    "Part 4 - latency logs",
    [
        check("GPU log valid (50+ records)",
              jsonLinesValid("~/edgeAILab/data/latencyGPU.jsonl",
                             requiredKeys=requiredRecordKeys, minRecords=50),
              hint="Re-run the GPU logger cell (DEVICE=0 ... latencyGPU.jsonl). If it crashed with a CUDA "
                   "error, your torch build does not match the Jetson stack - see Part 1.",
              link="https://jsonlines.org/"),
        check("CPU log valid (50+ records)",
              jsonLinesValid("~/edgeAILab/data/latencyCPU.jsonl",
                             requiredKeys=requiredRecordKeys, minRecords=50),
              hint="Re-run the CPU logger cell (DEVICE=cpu ... latencyCPU.jsonl).",
              link="https://jsonlines.org/"),
    ],
    successNote="Structured latency logs captured for CPU and GPU - the raw material for benchmarking.",
    docLink="https://docs.python.org/3/library/json.html",
)

---
## Part 5 · A Quick Look at the Numbers

For a fast sanity check on typical latency and FPS (the full statistical analysis — percentiles, tails, resource cost — is the job of the Benchmarking lab), run this **[Notebook cell]** directly:

In [ ]:
import json
import pathlib

for logName in ("latencyGPU.jsonl", "latencyCPU.jsonl"):
    logPath = pathlib.Path.home() / "edgeAILab" / "data" / logName
    if not logPath.exists():
        showNote(f"{logName} not found - run the Part 4 logger cells first.", kind="warn")
        continue
    endToEndValues = [json.loads(line)["latency_ms"]["end_to_end"]
                      for line in logPath.open() if line.strip()]
    meanLatency = sum(endToEndValues) / len(endToEndValues)
    print(f"{logName}:  runs: {len(endToEndValues)} | mean end-to-end: {meanLatency:.2f} ms "
          f"| approx FPS: {1000 / meanLatency:.1f}")

FPS is roughly `1000 / mean_latency_ms` for sequential inference. Hold on to these logs — you will analyze them properly in the Benchmarking lab.

### Checkpoint · Part 5

In [ ]:
import json
import pathlib

def latencyStagesProbe(fileName):
    # Verify the nested latency_ms stages the Benchmarking lab depends on.
    def probe():
        logPath = pathlib.Path.home() / "edgeAILab" / "data" / fileName
        if not logPath.is_file():
            return False, f"missing: {logPath}"
        firstLine = next((line for line in logPath.open() if line.strip()), None)
        if firstLine is None:
            return False, f"{fileName} is empty"
        stages = json.loads(firstLine).get("latency_ms", {})
        neededStages = {"preprocess", "inference", "postprocess", "end_to_end"}
        missing = neededStages - set(stages)
        return (not missing), ("all four latency stages present" if not missing
                               else f"missing stages: {sorted(missing)}")
    return probe

checkpoint(
    "Part 5 - latency schema sanity",
    [
        check("GPU log has all four latency stages", latencyStagesProbe("latencyGPU.jsonl"),
              hint="Re-run the %%writefile latencyLogger.py cell (Part 4) unchanged, then the GPU logger cell.",
              link="https://docs.ultralytics.com/modes/predict/#working-with-results"),
        check("CPU log has all four latency stages", latencyStagesProbe("latencyCPU.jsonl"),
              hint="Re-run the %%writefile latencyLogger.py cell (Part 4) unchanged, then the CPU logger cell.",
              link="https://docs.ultralytics.com/modes/predict/#working-with-results"),
    ],
    successNote="The logs carry preprocess / inference / postprocess / end_to_end - ready for real analysis.",
    docLink="https://docs.python.org/3/library/json.html",
)

---
## Part 6 · Run LocateAnything (Open-Vocabulary) Inference

YOLO detects a **fixed** set of classes it was trained on. A **LocateAnything**-style model is open-vocabulary: you give it a **text prompt** describing what to find, and it locates that, even for classes it was not explicitly trained on.

The exact open-vocabulary model and weights are **provided by your instructor** (for example, a YOLO-World or other promptable detector). The Part 1 setup cell already exported a default; adjust it here if needed:

In [ ]:
import os

# Replace the value with the model or weights your instructor provides.
os.environ["LOCATE_MODEL"] = os.environ.get("LOCATE_MODEL", "yolov8s-world.pt")
print("LOCATE_MODEL =", os.environ["LOCATE_MODEL"])

**[Notebook cell]** Create `locateAnything.py`:

In [ ]:
%%writefile locateAnything.py
import os
import json
import time
from datetime import datetime, timezone
from ultralytics import YOLO
from ultralytics.utils import ASSETS

modelName = os.environ["LOCATE_MODEL"]
deviceName = os.environ.get("DEVICE", "0")
deviceID = os.environ.get("DEVICE_ID", deviceName())
imagePath = os.environ.get("IMAGE", str(ASSETS / "bus.jpg"))
logFile = os.environ.get("LOG_FILE", "data/locateLatency.jsonl")

# promptText is what you want to locate, as a comma-separated list of phrases
promptText = os.environ.get("PROMPT", "person,bus,backpack")
classNames = [name.strip() for name in promptText.split(",") if name.strip()]

model = YOLO(modelName)

# open-vocabulary models accept the prompt as the set of classes to look for
if hasattr(model, "set_classes"):
    model.set_classes(classNames)

# warmup
model(imagePath, device=deviceName, verbose=False)

startTime = time.perf_counter()
results = model(imagePath, device=deviceName, verbose=False)
endTime = time.perf_counter()

result = results[0]
record = {
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "device_id": deviceID,
    "model": os.path.basename(str(modelName)),
    "task": "locate",
    "prompt": classNames,
    "compute": "gpu" if deviceName != "cpu" else "cpu",
    "latency_ms": {
        "preprocess": round(result.speed.get("preprocess", 0.0), 2),
        "inference": round(result.speed.get("inference", 0.0), 2),
        "postprocess": round(result.speed.get("postprocess", 0.0), 2),
        "end_to_end": round((endTime - startTime) * 1000, 2),
    },
    "num_detections": len(result.boxes),
}

os.makedirs(os.path.dirname(logFile) or ".", exist_ok=True)
with open(logFile, "a") as outFile:
    outFile.write(json.dumps(record) + "\n")

print(json.dumps(record, indent=2))
result.save(filename="data/locate.jpg")
print("annotated image saved to data/locate.jpg")

**[Notebook cell]** Run it with a prompt describing what to find:

In [ ]:
!PROMPT="person,backpack" DEVICE=0 python3 locateAnything.py

**[Notebook cell]** Try a different prompt on the same image and watch the detections change:

In [ ]:
!PROMPT="shoe,window" DEVICE=0 python3 locateAnything.py

In [ ]:
import pathlib
from IPython.display import Image as ipyImage, display

locatePath = pathlib.Path.home() / "edgeAILab" / "data" / "locate.jpg"
if locatePath.exists():
    display(ipyImage(filename=str(locatePath), width=480))
else:
    showNote("data/locate.jpg not found - run a locateAnything.py cell above first.", kind="warn")

Notice two things:

- the **same image** yields different detections depending on your text prompt
- open-vocabulary inference is usually **slower** than plain YOLO — compare the `inference` times. Flexibility costs latency, which is exactly the kind of trade-off you benchmark.

The latency record uses the same structure as the YOLO logger (with `task: "locate"` and a `prompt` field), so both feed the same analysis.

### Checkpoint · Part 6

In [ ]:
checkpoint(
    "Part 6 - open-vocabulary inference",
    [
        check("locate latency log valid",
              jsonLinesValid("~/edgeAILab/data/locateLatency.jsonl",
                             requiredKeys=["timestamp", "device_id", "model", "task",
                                           "prompt", "compute", "latency_ms", "num_detections"],
                             minRecords=2),
              hint="Run both PROMPT cells above. If loading LOCATE_MODEL failed, set the model your "
                   "instructor provided in the LOCATE_MODEL cell at the top of Part 6.",
              link="https://docs.ultralytics.com/models/yolo-world/"),
        check("annotated locate.jpg written", fileExists("~/edgeAILab/data/locate.jpg"),
              hint="Re-run a locateAnything.py cell above - the image is saved on every run.",
              link="https://docs.ultralytics.com/modes/predict/#working-with-results"),
    ],
    successNote="You steered a detector with plain text - and paid for that flexibility in latency.",
    docLink="https://docs.ultralytics.com/models/yolo-world/",
)

---
## Part 7 · Time a Realistic Workload

A single image is a microbenchmark. Real edge AI processes a **stream**. Simulate one by running inference repeatedly and watching live device load.

**[Notebook cell]** Write the load-watcher script for the terminal:

> **📟 On a Jetson:** telemetry tooling differs. Jetson's integrated Tegra GPU historically isn't exposed through `nvidia-smi`, so you watch it with **`tegrastats`** (or `jtop`). This **GB10** is a discrete GPU with a standard driver, so **`nvidia-smi dmon`** streams util / power / temperature. Note: GB10 has *unified* memory, so GPU vs. system memory blur — use `free -h` for the memory picture.

In [ ]:
%%writefile watchTegrastats.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeAILab/labEnv.sh

echo "Live GPU / memory load from EVERY user on this DGX Spark - press Ctrl-C to quit."
if command -v nvidia-smi >/dev/null; then
  nvidia-smi dmon        # GB10: util/power/temp; add `free -h` for unified memory
else
  tegrastats             # Jetson: integrated-GPU telemetry
fi

In [ ]:
!chmod +x watchTegrastats.sh

**[Terminal]** Start watching device load first (it streams forever — that is why it lives in a terminal):

```bash
~/edgeAILab/watchTegrastats.sh
```

**[Notebook cell]** Then start a longer run here — 100 GPU inferences, expect a few minutes (a full sustained-load and thermal analysis comes in the Benchmarking lab; here you just capture a stream log):

In [ ]:
!DEVICE=0 RUNS=100 LOG_FILE=data/latencyStream.jsonl python3 latencyLogger.py

Stop `tegrastats` with **Ctrl-C** in the terminal when the run finishes.

> Because the DGX Spark is shared, `tegrastats` shows GPU load from every student. If your inference latency is higher than expected, the GPU may be busy with someone else's work — a real property of shared edge hardware.

You now have a larger latency log captured under sustained load. Keep all three logs (`latencyGPU.jsonl`, `latencyCPU.jsonl`, `latencyStream.jsonl`) for the Benchmarking lab.

### Checkpoint · Part 7

In [ ]:
checkpoint(
    "Part 7 - stream workload captured",
    [
        check("stream log valid (100+ records)",
              jsonLinesValid("~/edgeAILab/data/latencyStream.jsonl",
                             requiredKeys=["timestamp", "device_id", "model", "task",
                                           "compute", "run", "latency_ms", "num_detections"],
                             minRecords=100),
              hint="Re-run the RUNS=100 logger cell above and let it finish.",
              link="https://jsonlines.org/"),
        check("all three logs on disk for the next lab",
              fileNonEmpty("~/edgeAILab/data/latencyCPU.jsonl", minLines=50),
              hint="latencyCPU.jsonl is missing or short - re-run the Part 4 CPU logger cell. The "
                   "Benchmarking lab needs all three logs.",
              link="https://jsonlines.org/"),
    ],
    successNote="Stream log captured under (shared) load - all inputs for the Benchmarking lab are ready.",
    docLink="https://docs.ultralytics.com/guides/nvidia-jetson/",
)

---
## Cleanup

Your logs and scripts are small, but model weights and the virtual environment can be large.

> **Keep `~/edgeAILab/data/*.jsonl` and `latencyLogger.py`** — the Benchmarking lab reads the logs and re-runs the logger. Clean up only after that lab is done.

If you used Option B in a terminal, `deactivate` leaves the venv (the notebook kernel keeps its own environment either way). If your instructor pre-staged shared model weights, do **not** delete those — only remove your own files. If you used a shared inference container, leave it running for other students unless told otherwise.

**[Notebook cell]** Optional · remove your lab folder (this also removes downloaded weights, the venv, **and the logs the next lab needs** — uncomment only when fully done):

In [ ]:
# !rm -rf ~/edgeAILab

---
### Lab scorecard

In [ ]:
labSummary("YOLO, LocateAnything & Latency Logging Lab")

---
## Part 8 · Connect to the Rest of the Course

You ran two styles of edge inference and logged their latency in a consistent format:

```text
Camera / Image
    -> YOLO or LocateAnything inference on the DGX Spark's GPU   (this lab)
    -> detections + structured latency log
    -> Benchmarking & Evaluation                            (next lab)
    -> Optimization (make it faster / smaller)              (later lab)
```

A real project could use this for distracted-driver detection, traffic monitoring, robotics perception, or industrial inspection. The main idea: running AI at the edge is about balancing accuracy, latency, and resources — and the first step is measuring latency honestly, with warmup, many runs, and a structured log.

---
## Key Terms

- **Edge AI / inference:** running a trained model on the device to get predictions, instead of sending data to the cloud.
- **YOLO:** a fast object detector that finds a *fixed* set of classes it was trained on.
- **Open-vocabulary detection ("locate anything"):** a model you steer with a text prompt, so it can locate things it wasn't explicitly trained to detect.
- **Latency:** how long one inference takes; the `speed` dict splits it into preprocess / inference / postprocess.
- **Throughput (FPS):** how many inferences per second; roughly `1000 / mean_latency_ms` for sequential runs.
- **Warmup:** discarding the first few inferences from measurement because they include one-time setup and run slower.
- **CPU vs. GPU (`device=cpu` vs `device=0`):** which processor runs the model; the GPU is the DGX Spark's accelerator and is much faster for the forward pass.